In [ ]:
import os, subprocess, urllib.request
BASE = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
r = subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True)
if r.returncode != 0 or "GPU" not in r.stdout:
    raise SystemExit("⛔ NO HAY GPU. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Entorno de ejecución → GPU T4.")
print("✅ GPU:", r.stdout.strip())
if "P100" in r.stdout: print("⚠️ Es una P100: mejor elige T4 x2.")
try: urllib.request.urlopen("https://huggingface.co", timeout=10); print("✅ Internet activado")
except Exception: raise SystemExit("⛔ SIN INTERNET. Kaggle: Settings → Internet → ON (requiere teléfono verificado).")


# Piso 3ºB · Prueba Qwen con descripciones en inglés (acento de España)
Genera **30 audios numerados** (6 personajes × 5 emociones) con Qwen3-TTS VoiceDesign, usando el estilo de descripción que sí te dio español de España:
*«Native voice from Spain… /θ/… no Latin American accent…»*.

Al terminar, descarga **qwen_ingles.zip** (Kaggle: pestaña Output; Colab: se descarga solo). Dime qué números suenan a España y con emoción.


In [ ]:
import json
os.makedirs(BASE + "/qwen_ingles", exist_ok=True)
ACENTO = ("Native voice from Spain (Peninsular Spanish, es-ES). Authentic Castilian Spanish pronunciation, with natural pronunciation of the letters Z and C "
          "before E/I using the Spanish /θ/ sound, and a clear distinction between S and Z. No Latin American accent, no Mexican, Colombian or Argentine pronunciation. "
          "Natural breathing, realistic and expressive delivery. Consistent vocal identity. ")
VOCES = {
 "yo":     "Male voice, about 22 years old, medium-low pitch, slightly hoarse. Dry, deadpan, ironic delivery of someone who is fed up with everything.",
 "amigo":  "Male voice, about 22 years old, relaxed, joking and very colloquial, warm and slightly deep.",
 "amiga":  "Female voice, about 22 years old, clear and energetic, quick-witted and sarcastic, bright tone.",
 "jefe":   "Male voice, about 55 years old, raspy and authoritative, a grumpy landlord. Medium-low pitch, not exaggeratedly deep, natural and human.",
 "abuela": "Female voice, about 75 years old, slightly shaky and cracked, a nosy but affectionate neighbour.",
 "madre":  "Female voice, about 50 years old, clearly feminine, warm and a little worried, a caring Spanish mother who worries about everything.",
}
EMO = {
 "neutro":   "Speaks naturally.",
 "cansado":  "Speaks very tired, slow and low, with a sigh, flat and drained.",
 "enfadado": "Speaks angry, loud, tense and fast.",
 "alegre":   "Speaks cheerful and excited, with energy and a smile in the voice.",
 "panico":   "Speaks in panic, very fast and high-pitched, almost shouting.",
}
FRASE = {
 "yo":     "Son las ocho y ocho, y ya llego tarde otra vez.",
 "amigo":  "Tranquilo, tío, que sobra tiempo. ¿Quién ha dejado la sartén en el suelo?",
 "amiga":  "No pasa nada, es puntualidad. Pero la nevera ha hecho un ruido muy raro.",
 "jefe":   "Esto es un piso, no un hotel. Quiero el alquiler antes del lunes, ¿vale?",
 "abuela": "Ay, hijo, qué ruido hacéis. En mis tiempos esto no pasaba.",
 "madre":  "Cariño, ¿has comido algo? Que te veo muy delgado, de verdad.",
}
SEM = {"yo": 11, "amigo": 21, "amiga": 31, "jefe": 41, "abuela": 51, "madre": 61}
trabajo, n = [], 0
for k in VOCES:
    for e in EMO:
        n += 1
        trabajo.append({"n": n, "k": k, "e": e, "seed": SEM[k], "text": FRASE[k], "instruct": ACENTO + VOCES[k] + " " + EMO[e], "out": f"{BASE}/qwen_ingles/{n:02d}_{k}_{e}.wav"})
json.dump(trabajo, open(BASE + "/trabajo.json", "w"))
open(BASE + "/gen.py", "w").write(f'''
import json, random, numpy as np, torch, soundfile as sf
from qwen_tts import Qwen3TTSModel
m = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.float16, attn_implementation="sdpa")
for t in json.load(open("{BASE}/trabajo.json")):
    random.seed(t["seed"]); np.random.seed(t["seed"]); torch.manual_seed(t["seed"]); torch.cuda.manual_seed_all(t["seed"])
    w, sr = m.generate_voice_design(text=t["text"], language="Spanish", instruct=t["instruct"])
    sf.write(t["out"], w[0], sr); print("Nº", t["n"], "/ 30", t["k"], t["e"], flush=True)
''')
QV = BASE + "/qv"
if not os.path.exists(QV + "/bin/python"):
    r0 = subprocess.run(f"pip install -q uv && uv venv --python 3.12 {QV} && uv pip install -q --python {QV}/bin/python qwen-tts soundfile", shell=True)
    if r0.returncode != 0 or not os.path.exists(QV + "/bin/python"):
        raise SystemExit("ERROR instalando Qwen (código %s). Copia el mensaje de arriba." % r0.returncode)
p = subprocess.Popen([QV + "/bin/python", "-u", BASE + "/gen.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
cola = []
for linea in p.stdout:
    cola.append(linea)
    if "Nº" in linea or "Error" in linea or "error" in linea: print(linea, end="")
p.wait()
if p.returncode != 0:
    print("\n===== ERROR EN QWEN (código %s). ÚLTIMAS LÍNEAS: =====" % p.returncode); print("".join(cola[-40:]))
else:
    subprocess.run(f"cd {BASE} && zip -qr qwen_ingles.zip qwen_ingles", shell=True)
    print("✅ Listo: qwen_ingles.zip")
    try:
        from google.colab import files; files.download(BASE + "/qwen_ingles.zip")
    except Exception: pass
